# Final Implementation Runs for Markov Chain Classification of Genomic Features Over Varying k Values

This notebook chronicles the Final Execution of this project. It is a culmination of work spanning four months that critically investigates a Markov Chain model's ability to recognize patterns unique to each genomic feature and then use that information to distinguish between four canonical classes: promoters, exons, introns, and repeats. The model will be evaluated under different orders of k to identify which order is optimal for each class and at what k value(s) the model breaks down.

### Set Working Directory to Root and Add src/ and notebooks/ to Python Path

In [ ]:
# Set working directory to root and add src/ and notebooks/ to path
import os
import sys

from data_preparation import load_fasta

cwd = os.getcwd()
# Ensure working dir is root, not src/ or notebooks/
if cwd.endswith("src") or cwd.endswith("notebooks"):
    os.chdir("..")

print("Working directory:", os.getcwd())

# Add src/ to Python path
sys.path.insert(0, "src")
sys.path.insert(0, "notebook")

### Import Packages and Submodules Required to Execute Programs for Final Run

The following cell imports all program scripts required to execute a full runthrough of the model, including loading the datafiles into memory, training the model, calculating thresholds, testing, generating classifications, calculating per-base log-likelihood scores, evaluating results, and visualizing results. All scripts have been written to run exclusively within this notebook with the provided calls. While technically, the scripts can be run individually from a local terminal, this would require either including the entirety of a call in the terminal, or modifying the scripts themselves to manually reproduce the full workflow, output structure, and visualization pipeline used here. In practice, this would involve rewriting large portions of the driver logic, restructuring return objects, and adding additional I/O handling to generate results in the same format produced automatically within the notebook. For this reason, the notebook serves as the intended and most efficient execution environment for the complete model pipeline.

In [ ]:
# Import packages and submodules
from evaluate_results import compute_confusion_matrix, compute_per_class_accuracy, compute_overall_accuracy, compute_class_metrics
from per_base_likelihood import per_base_likelihood
from threshold_utils import compute_thresholds
from model_classification import (
    classify_sequence,
    classify_with_reject,
    generate_full_results
)
from data_loading import read_fasta, load_class_seqs
from training_utils import train_markov_model, train_all_models

### Define Training Data FASTA Path for Final Run and Load into Dictionary

In [ ]:
# Define datapaths for input training and testing datasets
training_datapath = "data/final/train/train.fasta"
testing_datapath = "data/final/test/test.fasta"

### Load Training Dataset for Each Class into Model from Single, Combined File

In [ ]:
training_data = load_fasta(training_datapath)


### Train Markov Model Over Increasing k Values to Generate Transition Probabilities and Threshold Values for Testing Phase

In this step the model is trained with the training data, which contains the sequences for each class and their associated true labels. For k = 3, there are 4^3 = 64 possible states and for 10x coverage to ensure reliable results, we are aiming for 10 observations per state, which equals a total of 640 3-mers. 640 3-mers comes from: L - 3 + 1 = 640 -> 642 total basepairs, meaning we need at least 642 total basepairs in a class to have 10x coverage for the 3-mers. Since our sequences

In [ ]:
# Train Markov models for structural classes over k = 1, 2, 3, 4, 5

k_values_final = [1, 2, 3, 4, 5]

all_models_final_run = {}       # mapping: k -> {class_label -> model}
all_thresholds_final_run = {}   # mapping: k -> {class_label -> threshold}

for k in k_values_final:
    # Train models for this k
    models = train_all_models(training_data, k)
    all_models_final_run[k] = models

    # Compute thresholds for this k
    thresholds_k_final_run = compute_thresholds(training_data, models, k)
    all_thresholds_final_run[k] = thresholds_k_final_run

# Display transition probabilities and thresholds for verification
all_models_final_run, all_thresholds_final_run


### Define Test Inapath and Prepare Sequences and True Labels to Test Model

###

### Calculate Per-k and Per-Class Results

In [ ]:
def evaluate_by_k(results_by_k, class_labels):
    """
    Compute accuracy and class metrics for each k value.
    Parameters:
        results_by_k (dict[int, list[dict]]): mapping k → list of result dicts
        class_labels (list[str]): ordered list of class names
    Returns:
        dict[int, dict]: mapping k → metrics including confusion matrix,
                         overall accuracy, per-class accuracy, and class metrics
    """
    summary = {}

    for k, results in results_by_k.items():
        # Convert results into (true, pred, seq) tuples for existing functions
        formatted = [(r["true_label"], r["predicted_label"], None) for r in results]

        conf = compute_confusion_matrix(formatted, class_labels)
        overall = compute_overall_accuracy(conf)
        per_class_acc = compute_per_class_accuracy(conf, class_labels)
        class_metrics = compute_class_metrics(conf, class_labels)

        summary[k] = {
            "confusion_matrix": conf,
            "overall_accuracy": overall,
            "per_class_accuracy": per_class_acc,
            "class_metrics": class_metrics
        }

    return summary
